In [1]:
print("h"*10)

hhhhhhhhhh


In [2]:
 
#!/usr/bin/env python3
"""
check_local_llm_capability.py
Detects your laptop's RAM, CPU, and GPU/VRAM, then recommends which
open-source LLMs (via Ollama / llama.cpp GGUF quantization) you can
run smoothly.
Usage:
    pip install psutil --break-system-packages   # if not already installed
    python3 check_local_llm_capability.py
Notes:
- GPU/VRAM detection works for NVIDIA (via nvidia-smi) and Apple Silicon
  (unified memory = RAM). AMD/Intel GPU VRAM detection is best-effort.
- "Smooth" here means the model's quantized size comfortably fits in
  available RAM/VRAM with headroom for context + OS overhead.
"""
import platform
import shutil
import subprocess
import psutil
# ---- Model catalog: (name, params, recommended min RAM/VRAM in GB for Q4_K_M quant) ----
# Sizes are approximate for 4-bit quantized (Q4_K_M) GGUF versions.
MODEL_CATALOG = [
    # (display name, approx quantized size GB, min RAM needed GB, tier)
    ("DeepSeek-R1-Distill 1.5B",         1.1, 3,  "Tiny Reasoning (very fast)"),
    ("Qwen2.5 / Llama 3.2 1B-3B",        2.0, 4,  "Tiny (fast, lower quality)"),
    ("Phi-3.5-mini / Gemma2 2B",         3.0, 6,  "Small (good on modest laptops)"),
    ("Qwen2.5-Coder 7B",                 4.7, 8,  "Medium Coding (excellent logic)"),
    ("DeepSeek-R1-Distill 8B",           4.9, 8,  "Medium Reasoning (great general logic)"),
    ("Llama 3.1 8B / Mistral 7B",        5.0, 8,  "Medium (solid general use)"),
    ("Mistral-Nemo 12B",                 7.5, 10, "Medium-Large (long context)"),
    ("Qwen2.5 14B / Gemma2 9B / Phi-4",  9.0, 16, "Medium-Large (high quality)"),
    ("DeepSeek-R1-Distill 14B",          9.0, 16, "Medium-Large Reasoning (smart, balanced)"),
    ("DeepSeek-R1-Distill 32B",          20.0, 24, "Large Reasoning (advanced math/logic)"),
    ("Mixtral 8x7B (MoE)",               26.0, 32, "Large MoE (needs strong RAM/VRAM)"),
    ("Llama 3.3 70B / Qwen2.5 72B",      42.0, 48, "Very Large (workstation-class)"),
    ("DeepSeek-R1-Distill 70B",          42.0, 48, "Very Large Reasoning (near frontier)"),
    ("Llama 3.1 405B",                  230.0, 256, "Frontier-scale (multi-GPU server)"),
    ("DeepSeek-R1 (671B MoE)",          400.0, 512, "Frontier MoE (multi-node datacenter)"),
]

def get_ram_gb():
    return round(psutil.virtual_memory().total / (1024 ** 3), 1)

def get_cpu_info():
    return platform.processor() or platform.machine(), psutil.cpu_count(logical=False), psutil.cpu_count(logical=True)

def get_disk_free_gb(path="/"):
    total, used, free = shutil.disk_usage(path)
    return round(free / (1024 ** 3), 1)

def get_gpu_info():
    """Best-effort GPU + VRAM detection. Returns (gpu_name, vram_gb) or (None, 0)."""
    system = platform.system()
    # Apple Silicon: unified memory acts as VRAM
    if system == "Darwin" and platform.machine() == "arm64":
        try:
            chip = subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], text=True
            ).strip()
        except Exception:
            chip = "Apple Silicon"
        return f"{chip} (Unified Memory)", get_ram_gb()
    # NVIDIA via nvidia-smi
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                text=True,
            ).strip()
            name, mem_mb = out.split(",")
            return name.strip(), round(float(mem_mb.strip()) / 1024, 1)
        except Exception:
            pass
    return None, 0

def recommend_models(effective_gb):
    """Return list of models that fit within effective available memory, with headroom."""
    usable = effective_gb * 0.7  # leave ~30% headroom for OS/context/other apps
    fits, tight, wont_fit = [], [], []
    for name, size_gb, min_ram, tier in MODEL_CATALOG:
        if size_gb <= usable:
            fits.append((name, size_gb, tier))
        elif size_gb <= effective_gb:
            tight.append((name, size_gb, tier))
        else:
            wont_fit.append((name, size_gb, tier))
    return fits, tight, wont_fit

def main():
    import sys
    import io
    if sys.stdout.encoding != 'utf-8':
        try:
            sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
        except Exception:
            pass
    print("=" * 60)
    print("Local LLM Capability Check")
    print("=" * 60)
    system = platform.system()
    ram_gb = get_ram_gb()
    proc, phys_cores, logical_cores = get_cpu_info()
    disk_free = get_disk_free_gb()
    gpu_name, vram_gb = get_gpu_info()
    print(f"\nOS:            {system} {platform.release()}")
    print(f"CPU:           {proc} ({phys_cores} physical / {logical_cores} logical cores)")
    print(f"RAM:           {ram_gb} GB")
    print(f"Free Disk:     {disk_free} GB")
    if gpu_name:
        print(f"GPU:           {gpu_name}")
        print(f"VRAM:          {vram_gb} GB")
    else:
        print("GPU:           No dedicated GPU detected (CPU-only inference)")
    # Effective memory pool for model weights:
    # - If a real GPU with VRAM was found, use VRAM (best case, fastest).
    # - Otherwise fall back to system RAM (CPU inference via llama.cpp).
    if gpu_name and vram_gb > 0:
        effective_gb = vram_gb
        mode = "GPU-accelerated"
    else:
        effective_gb = ram_gb
        mode = "CPU-only"
    print(f"\nInference mode: {mode}")
    print(f"Usable memory for model weights: ~{effective_gb} GB")
    fits, tight, wont_fit = recommend_models(effective_gb)
    print("\n--- Should run SMOOTHLY (Q4_K_M quantization) ---")
    if fits:
        for name, size_gb, tier in fits:
            print(f"  ✅ {name:<32} ~{size_gb} GB  [{tier}]")
    else:
        print("  None comfortably — consider the tightest-fit models below with caution.")
    print("\n--- Will run but TIGHT (little headroom, may be slow / risk OOM) ---")
    if tight:
        for name, size_gb, tier in tight:
            print(f"  ⚠️  {name:<32} ~{size_gb} GB  [{tier}]")
    else:
        print("  None")
    print("\n--- Won't fit ---")
    for name, size_gb, tier in wont_fit:
        print(f"  ❌ {name:<32} ~{size_gb} GB  [{tier}]")
    if disk_free < 10:
        print("\n⚠️  Low disk space — make sure you have enough room to download model files.")
    print("\nTip: Run these via Ollama (https://ollama.com) — e.g. `ollama run llama3.1:8b`")
    print("     or llama.cpp with GGUF files for finer control over quantization.")
    print("=" * 60)

if __name__ == "__main__":
    main()
 

Local LLM Capability Check

OS:            Windows 10
CPU:           Intel64 Family 6 Model 154 Stepping 3, GenuineIntel (8 physical / 12 logical cores)
RAM:           15.7 GB
Free Disk:     13.5 GB
GPU:           NVIDIA GeForce RTX 3050 Laptop GPU
VRAM:          4.0 GB

Inference mode: GPU-accelerated
Usable memory for model weights: ~4.0 GB

--- Should run SMOOTHLY (Q4_K_M quantization) ---
  ✅ DeepSeek-R1-Distill 1.5B         ~1.1 GB  [Tiny Reasoning (very fast)]
  ✅ Qwen2.5 / Llama 3.2 1B-3B        ~2.0 GB  [Tiny (fast, lower quality)]

--- Will run but TIGHT (little headroom, may be slow / risk OOM) ---
  ⚠️  Phi-3.5-mini / Gemma2 2B         ~3.0 GB  [Small (good on modest laptops)]

--- Won't fit ---
  ❌ Qwen2.5-Coder 7B                 ~4.7 GB  [Medium Coding (excellent logic)]
  ❌ DeepSeek-R1-Distill 8B           ~4.9 GB  [Medium Reasoning (great general logic)]
  ❌ Llama 3.1 8B / Mistral 7B        ~5.0 GB  [Medium (solid general use)]
  ❌ Mistral-Nemo 12B                 ~7.5 